# Waterberry Farms: a realistic benchmark for multi-robot informative path planning

This notebook generates the figures and tables of the paper "Waterberry Farms: A Novel Benchmark For Informative Path Planning" by
Samuel Matloob, O. Patrick Kreidl, Ayan Dutta, Swapnoneel Roy, Ladislau Bölöni. (https://arxiv.org/abs/2305.06243)

It illustrates various ways of running a __one day__ experiment on a __static environment__ with a __single robot__. The experiments compare estimator/path planner pairs. The path planners are simple static examples (lawnmower, spiral, random waypoint) and they are not intended to be state of the art.

The figures of the estimator comparison are made from the results of the `estimator-benchmark/miniberry-30-all` exp/run, which must be run first with `notebooks/EstimatorBenchmark-Run.ipynb`.


In [ ]:
# allow imports from the main source directory
import sys
sys.path.append('../..')
import logging
logging.basicConfig(level=logging.WARNING)

import pathlib
import pickle
import gzip as compress
import copy
import matplotlib.pyplot as plt
# configuring the fonts such that no Type 3 fonts are used
# requirement for ICRA
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams["figure.autolayout"] = True

import numpy as np
import timeit
import warnings

from exp_run_config import Config
import pandas as pd
from water_berry_farm import WBF_Score_WeightedAsymmetric
from information_model import DISEASED
from wbf_helper import create_estimator, create_wbf, create_wbfe, get_geometry, precompute_environment
from wbf_simulate import save_simulation_results, simulate_1day
from wbf_figures import end_of_day_graphs, hook_create_pictures
from policy import FollowPathPolicy, RandomWaypointPolicy
from path_generators import find_fixed_budget_lawnmower, find_fixed_budget_spiral, generate_lawnmower, get_path_length
from robot import Robot
Config.PROJECTNAME = "WaterBerryFarms"

logging.getLogger().setLevel(logging.WARNING)

# all the experiments results etc. should go into this directory
benchmark_dir = pathlib.Path(Config().get_results_path(), "Benchmark-Paper")
benchmark_dir.mkdir(parents=True, exist_ok=True)

## General settings
The choices used here apply to all the components and might make them faster

In [ ]:
# the environment exp/run used for each farm type
environment_runs = {"Miniberry-10": "benchmark-miniberry-10", "Miniberry-30": "benchmark-miniberry-30",
                    "Miniberry-100": "benchmark-miniberry-100", "Waterberry": "benchmark-waterberry"}
# the farm type of the one day experiments
experiment_typename = "Miniberry-30"
timesteps = 500 # 1000
time_start_environment = 6
# change this to a lower number to run the estimator more often
estimator_interval = 10
# if force_run is false, we do not re-run an already run experiment
force_run = False

def run_1day(policy, estimator, evaluator):
    """Runs a one day experiment with a single robot on the experiment farm and saves the results into the 
    benchmark directory. If the results already exist and force_run is false, it loads them instead."""
    resultsfile = pathlib.Path(benchmark_dir, f"res_{policy.name}-{estimator.name}.pickle")
    print(resultsfile)
    if resultsfile.exists() and not force_run:
        with compress.open(resultsfile, "rb") as f:
            return pickle.load(f)
    _, environment = create_wbfe(Config().get_experiment("environment", environment_runs[experiment_typename]))
    environment.proceed(time_start_environment)
    robot = Robot("Rob", 0, 0, 0)
    robot.assign_policy(policy)
    # add after_timestep = hook_create_pictures to generate pictures. WARNING: takes longer!
    results = simulate_1day(environment=environment, robots=[robot], estimator=estimator, evaluator=evaluator,
                            timesteps=timesteps, estimator_interval=estimator_interval)
    results.update({"wbfe": environment, "policy-name": policy.name, "estimator-name": estimator.name,
                    "results-basedir": benchmark_dir})
    save_simulation_results(resultsfile, results)
    return results

def make_estimator(estimator_run, geo):
    """A fresh estimator from its estimator exp/run"""
    return create_estimator(Config().get_experiment("estimator", estimator_run, create_data_dir=False), geo)


# Precompute environments
* Precompute the evaluation of the environments. This allows the following simulations to run faster and to use the same environment for different path planning algorithms. 

* Each farm type has its own environment exp/run (`data/expruns/environment/benchmark-*.yaml`), which precalculates 50 days of evolution. This is sufficient for the standard benchmark settings. An environment is precomputed only once: it is skipped if an earlier precomputation completed. __Delete its directory `environment/benchmark-*` in the results path if you want to recompute it.__

* The full Waterberry farm is not cached (its cache would take about 36 GB), so it is not precomputed: the cells below that use Waterberry compute its environment live, which takes long for the later days. Set `cache: true` in `benchmark-waterberry.yaml` to cache it instead.

In [ ]:
# Waterberry is not cached, so precompute_environment would do nothing for it
typenames = ["Miniberry-10", "Miniberry-30", "Miniberry-100"]
for typename in typenames:
    precompute_environment(environment_runs[typename])

# Generate pictures of the geometry. 
Generates pictures for the geometry of the various benchmark configurations (Miniberry-10,30,100 and Waterberry), and saves them into files $data/geometry_Miniberry-10.pdf etc. Only the farm geometry is created, so this does not need the precomputed environments.

In [ ]:
typenames_1 = ["Miniberry-10", "Miniberry-30", "Miniberry-100", "Waterberry"]
for typename in typenames_1:
    wbf = create_wbf({"typename": typename})
    fig, ax = plt.subplots(1, figsize=(5,4))
    wbf.visualize(ax)
    fig.tight_layout()
    plt.savefig(pathlib.Path(benchmark_dir, f"geometry_{typename}.pdf"))
    plt.savefig(pathlib.Path(benchmark_dir, f"geometry_{typename}.jpg"))

## Compare movement policies and estimators
Run one day experiments for every combination of the four movement policies with the adaptive disk (AD) and the kriging (KRG) estimators. Visualize the results for each combination.


In [ ]:
def create_policies(geom):
    policies = []
    # a random waypoint policy
    geo = copy.copy(geom)
    policy = RandomWaypointPolicy(vel = 1, low_point = [geo["xmin"], geo["ymin"]], high_point = [geo["xmax"], geo["ymax"]], seed = 0)  
    policy.name = "RandomWaypoint-Seed0"
    policies.append(policy)

    # a lawnmower policy that covers the target area in one day uniformly
    geo = copy.copy(geom)
    path = find_fixed_budget_lawnmower([0,0], geo["xmin"], geo["xmax"], geo["ymin"], geo["ymax"], geo["velocity"], time = geo["timesteps-per-day"])
    policy = FollowPathPolicy(vel = geo["velocity"], waypoints = path, repeat = True)
    policy.name = "FixedBudgetLawnmower"
    policies.append(policy)

    # a lawnmower policy that covers the tomatoes more densely than the strawberries
    geo = copy.copy(geom)
    height = geo["ymax"] - geo["ymin"]
    time = float(geo["timesteps-per-day"])
    time_safety = time / 10.0
    # the strawberry leg gets a third of the time, but at least enough for a one-wind lawnmower 
    # from any corner of the tomato area where the tomato leg may end
    strawberry_area = [geo["xmin"], geo["xmax"], geo["ymin"], geo["ymin"] + height // 2]
    corners = [[x, y] for x in [geo["xmin"], geo["xmax"]] for y in [geo["ymin"] + height // 2, geo["ymin"] + height]]
    time_strawberry_min = max(get_path_length(generate_lawnmower(*strawberry_area, 1), corner) for corner in corners) / geo["velocity"]
    time_strawberry = max(time / 3.0 - time_safety, time_strawberry_min)
    time_tomato = time - time_strawberry - time_safety
    path1 = find_fixed_budget_lawnmower([0, 0], geo["xmin"], geo["xmax"], geo["ymin"] + height//2, geo["ymin"] + height, geo["velocity"], time_tomato)
    path2 = find_fixed_budget_lawnmower(path1[-1], geo["xmin"], geo["xmax"], geo["ymin"], geo["ymin"] + height // 2, geo["velocity"], time_strawberry)
    # second path is supposed to start from the last of first
    path = np.append(path1, path2, axis=0)           
    policy = FollowPathPolicy(vel = geo["velocity"], waypoints = path, repeat = True)            
    policy.name = "AdaptiveLawnMower"
    policies.append(policy)

    # a spiral policy 
    geo = copy.copy(geom)
    path = find_fixed_budget_spiral([0,0], x_min = geo["xmin"], x_max = geo["xmax"], y_min = geo["ymin"], y_max = geo["ymax"], velocity = geo["velocity"], time = geo["timesteps-per-day"])
    policy = FollowPathPolicy(vel = geo["velocity"], waypoints = path, repeat = True) 
    policy.name = "Spiral"    
    policies.append(policy)
    return policies

In [ ]:
geo = get_geometry(experiment_typename)
for estimator_run in ["adaptive-disk", "gp-kriging"]:
    for policy in create_policies(geo):
        results = run_1day(policy, make_estimator(estimator_run, geo), WBF_Score_WeightedAsymmetric())
        end_of_day_graphs(results, f"daily-summary-{results['policy-name']}-{results['estimator-name']}.pdf", plot_uncertainty = False)


## The canonical Gaussian Process
The Gaussian Process in its canonical form, with a prior mean of 0, combined with the random waypoint policy. It is much slower, so it will take a quite longer time.


In [ ]:
# cleaning up the warnings from the GP estimator
warnings.simplefilter("ignore")

geo = get_geometry(experiment_typename)
policy = RandomWaypointPolicy(vel = 1, low_point = [geo["xmin"], geo["ymin"]], high_point = [geo["xmax"], geo["ymax"]], seed = 0)
policy.name = "RandomWaypoint-Seed0"
results = run_1day(policy, make_estimator("gaussian-process", geo), WBF_Score_WeightedAsymmetric())
end_of_day_graphs(results, f"daily-summary-{results['policy-name']}-{results['estimator-name']}.pdf", plot_uncertainty = False)


## Compare the performance of various combinations of path planner and estimator

In [ ]:
# gather all the results of the policy / estimator comparison
allresults = []
for a in sorted(benchmark_dir.iterdir()):
    if a.name.startswith("res_") and not a.name.startswith("res_random-"):
        with compress.open(a, "rb") as f:
            allresults.append(pickle.load(f))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, estimator_name in zip(axes, ["AD", "KRG", "GP"]):
    for results in allresults:
        if results["estimator-name"] == estimator_name:
            events = results["score-events"]
            ax.plot([event["timestep"] for event in events], [event["score"] for event in events],
                    label = results["policy-name"])
    ax.set_xlabel("Time")
    ax.set_title(estimator_name)
    ax.legend(fontsize=8)
axes[0].set_ylabel("Score")
# AD and KRG on the same scale
axes[1].sharey(axes[0])
fig.tight_layout()
plt.savefig(pathlib.Path(benchmark_dir, "planner-estimator-scores.pdf"))

# the final scores
pd.Series({f'{r["policy-name"]}+{r["estimator-name"]}': r["score-events"][-1]["score"] for r in allresults}).round(2)


## Compare the results of different score functions
Verifying that different scoring functions are called

In [ ]:
geo = get_geometry(experiment_typename)

# create a set of score functions
scores = {}
scores["default"] = WBF_Score_WeightedAsymmetric()
scores["only-strawberry"] = WBF_Score_WeightedAsymmetric(strawberry_importance=1.0, strawberry_negative_importance=10.0, tomato_importance=0.0, tomato_negative_importance=10.0, soil_importance=0.0)
scores["only-tomato"] = WBF_Score_WeightedAsymmetric(strawberry_importance=0.0, strawberry_negative_importance=10.0, tomato_importance=1.0, tomato_negative_importance=10.0, soil_importance=0.0)
scores["only-soil"] = WBF_Score_WeightedAsymmetric(strawberry_importance=0.0, strawberry_negative_importance=10.0, tomato_importance=0.0, tomato_negative_importance=10.0, soil_importance=1.0)

for scorename, evaluator in scores.items():
    # the same random waypoint path for every score function
    policy = RandomWaypointPolicy(vel = 1, low_point = [geo["xmin"], geo["ymin"]], high_point = [geo["xmax"], geo["ymax"]], seed = 0)  
    policy.name = f"random-{scorename}"
    estimator = make_estimator("adaptive-disk", geo)
    results = run_1day(policy, estimator, evaluator)
    end_of_day_graphs(results, f"daily-summary-{results['policy-name']}-{results['estimator-name']}.pdf", plot_uncertainty = False)

# Comparing the computational cost of different estimators
This code measures the wall-clock time of a single estimate of each estimator, against the number of random observations, on the Miniberry-30 and Miniberry-100 farms. On the full Waterberry farm, only the estimators whose cost does not depend on the size of their training set are measured. An estimator is stopped once an estimate takes longer than `stop_at` seconds.


In [ ]:
logging.getLogger().setLevel(logging.WARNING)
warnings.simplefilter("ignore")

ESTIMATORS = ["point", "adaptive-disk", "nearest", "idw", "rbf", "gaussian-process", "gp-kriging", "gp-local",
              "gp-indicator", "occupancy", "mrf", "epidemic-pf", "cnn"]
cost_estimators = {"Miniberry-30": ESTIMATORS, "Miniberry-100": ESTIMATORS,
                   "Waterberry": ["point", "adaptive-disk", "nearest", "idw"]}

file = pathlib.Path(benchmark_dir, "timecomp.pickle")
checkpoints = [1, 10, 20, 50, 100, 200, 500] # the number of observations at which we measure
stop_at = 10.0 # if a single estimate takes longer than this, we stop

if file.exists():
    with open(file, "rb") as f:
        res = pickle.load(f)
else:
    res = {}
    for typename, estimator_runs in cost_estimators.items():
        geo = get_geometry(typename)
        _, wbfe = create_wbfe(precompute_environment(environment_runs[typename]))
        wbfe.proceed(time_start_environment)
        random = np.random.default_rng(0)
        positions = [[int(random.integers(geo["xmin"], geo["xmax"] + 1)), int(random.integers(geo["ymin"], geo["ymax"] + 1)), t]
                     for t in range(checkpoints[-1])]
        observations = [wbfe.get_observation(position) | {"robot": "random"} for position in positions]
        for estimator_run in estimator_runs:
            wbfim = make_estimator(estimator_run, geo)
            ts, times, added = [], [], 0
            for checkpoint in checkpoints:
                for observation in observations[added:checkpoint]:
                    wbfim.add_observation(observation)
                added = checkpoint
                time = timeit.timeit("wbfim.proceed(1)", number=1, globals=globals())
                print(f"{typename} {estimator_run} {checkpoint} = {time:.3f}")
                ts.append(checkpoint)
                times.append(time)
                if time > stop_at:
                    break
            res[(typename, estimator_run)] = [ts, times]
    with open(file, "wb") as f:
        pickle.dump(res, f)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, typename in zip(axes, cost_estimators):
    for k, estimator_run in enumerate(ESTIMATORS):
        if (typename, estimator_run) in res:
            ts, times = res[(typename, estimator_run)]
            ax.plot(ts, times, marker=".", label=estimator_run, color=plt.cm.tab20(k))
    ax.axhline(stop_at, color="black", linewidth=0.5, linestyle=":")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Observations")
    ax.set_title(typename)
axes[0].set_ylabel("Time (seconds)")
axes[0].legend(fontsize=7, ncol=2)
fig.tight_layout()
plt.savefig(pathlib.Path(benchmark_dir, "estimator_computational_cost.pdf"))


# Evolution of the environments at a list of specific timepoints

Generates figures for the state of the environment components at various timepoints. The default timepoints are the 1, 5, 10, 15 that are good for visualizing what goes on in the multiday experiments. 

The results are saved into $data/env-dynamics-{typename}-{value} files.

In [ ]:
typenames = ["Miniberry-100", "Waterberry"]
values = ["tylcv", "ccr", "soil"]
timepoints = [1, 5, 10, 15]

for value in values:
    for typename in typenames:
        _, wbfe = create_wbfe(precompute_environment(environment_runs[typename]))
        fig, axs = plt.subplots(1, len(timepoints), figsize = (2.5 * len(timepoints), 2.5))
        #fig.suptitle(f"{typename} / {value}")
        ntt = 0
        for t in range(timepoints[-1]+1):
            if t >= timepoints[ntt]:
                if value == "tylcv":
                    val = wbfe.tylcv.value
                elif value == "ccr":
                    val = wbfe.ccr.value
                elif value == "soil":
                    val = wbfe.soil.value
                axs[ntt].imshow(val.T, vmin=0, vmax=1, cmap="gray", origin="lower")
                axs[ntt].set_title(f"t={timepoints[ntt]}")
                ntt = ntt + 1
            wbfe.proceed()
        fig.tight_layout()
        plt.savefig(pathlib.Path(benchmark_dir, f"env-dynamics-{typename}-{value}.pdf"))
        plt.savefig(pathlib.Path(benchmark_dir, f"env-dynamics-{typename}-{value}.jpg"))

# The estimator benchmark

Figures and tables from the results of the `estimator-benchmark/miniberry-30-all` exp/run: all the estimators, on three scenarios, with observations from random samples and from the trajectories of a random waypoint and a lawnmower robot, 5 repetitions each.


In [ ]:
exp_bench = Config().get_experiment("estimator-benchmark", "miniberry-30-all", create_data_dir=False)
bench_dir = pathlib.Path(exp_bench["data_dir"])
bench = pd.read_pickle(pathlib.Path(bench_dir, "results.pickle"))
# the estimators shown in the figures; the tables show all of them
SELECTION = ["adaptive-disk", "idw", "gaussian-process", "gp-kriging", "gp-indicator", "mrf", "epidemic-pf", "cnn"]
SAMPLERS = {"random-500": "random samples", "random-waypoint-500": "random waypoint", "lawnmower-500": "lawnmower"}
SCENARIOS = {"miniberry-30": "Miniberry-30", "miniberry-30-tomato": "Miniberry-30 tomato-only", "mrmr-custom-clustered": "Miniberry-100 picture"}
tylcv = bench[bench.field == "tylcv"].groupby(["environment", "sampler", "estimator", "observations", "metric"]).value.mean().unstack("metric")


## Accuracy and recall against the number of observations
The asymmetric error (the TYLCV component of the score, with a missed disease weighted 10) and the recall of the diseased cells, on the standard Miniberry-30 scenario.


In [ ]:
metrics = [("asymmetric-error", "asymmetric error"), ("recall", "recall")]
fig, axes = plt.subplots(len(metrics), len(SAMPLERS), figsize=(12, 5.5), sharex=True, sharey="row")
for i, (metric, label) in enumerate(metrics):
    for j, (sampler, sampler_label) in enumerate(SAMPLERS.items()):
        ax = axes[i, j]
        for k, run in enumerate(SELECTION):
            curve = tylcv.loc[("miniberry-30", sampler, run)][metric]
            ax.plot(curve.index, curve.values, marker=".", label=run, color=plt.cm.tab10(k))
        ax.set_xscale("log")
        if i == 0:
            ax.set_title(sampler_label)
        if j == 0:
            ax.set_ylabel(label)
        if i == len(metrics) - 1:
            ax.set_xlabel("observations")
axes[1, 0].legend(fontsize=7, ncol=2)
fig.tight_layout()
plt.savefig(pathlib.Path(benchmark_dir, "estimators-accuracy.pdf"))


## The final metrics
After 500 observations, the mean over the repetitions. The table is also written as a LaTeX tabular into the benchmark directory.


In [ ]:
def latex_table(frame, filename, digits=2):
    """Writes a frame with an index and simple columns as a LaTeX tabular body"""
    lines = [" & ".join([frame.index.name] + list(frame.columns)) + " \\\\", "\\hline"]
    for index, row in frame.iterrows():
        lines.append(" & ".join([ABBREVIATIONS[index]] + ["--" if np.isnan(v) else f"{v:.{digits}f}" for v in row]) + " \\\\")
    pathlib.Path(benchmark_dir, filename).write_text("\n".join(lines) + "\n")

order = ["point", "adaptive-disk", "nearest", "idw", "rbf", "gaussian-process", "gp-kriging", "gp-local",
         "gp-indicator", "occupancy", "mrf", "epidemic-pf", "cnn"]
# the estimators are shown in the tables by their names, e.g. AD for adaptive-disk
ABBREVIATIONS = {run: Config().get_experiment("estimator", run, create_data_dir=False)["estimator-name"] for run in order}
final = tylcv.xs(500, level="observations")
for scenario in SCENARIOS:
    columns = {}
    for sampler, short in [("random-500", "rnd"), ("random-waypoint-500", "rwp")]:
        f = final.loc[(scenario, sampler)].reindex(order)
        columns[f"AE {short}"] = f["asymmetric-error"]
        columns[f"R {short}"] = f["recall"]
    table = pd.DataFrame(columns)
    table.index.name = "estimator"
    print(SCENARIOS[scenario])
    print(table.round(3))
    latex_table(table, f"table-accuracy-{scenario}.tex")


## Calibration of the confidence
The calibration metrics of the probability of disease (native, or derived from the value as `clip(2 (1 - value))`), the rank correlation between the uncertainty and the error, and the interval coverage of the estimators with a standard deviation. On the standard Miniberry-30 scenario with the random waypoint robot, after 500 observations.


In [ ]:
columns = ["brier", "ece", "uncertainty-error-correlation", "coverage-1sd", "coverage-2sd"]
calibration = final.loc[("miniberry-30", "random-waypoint-500")].reindex(order)[columns]
calibration.columns = ["Brier", "ECE", "UE corr.", "cov. 1sd", "cov. 2sd"]
calibration.index.name = "estimator"
latex_table(calibration, "table-calibration.tex")
calibration.round(3)


Reliability diagrams of the estimators with a native probability of disease, on all the tomato cells after the 500 observations of each regime, pooled over the repetitions and the two Miniberry-30 scenarios. Bins with fewer than 20 cells are not shown. The observation sets are the ones saved by the benchmark.


In [ ]:
geo = get_geometry("Miniberry-30")
environments = {}
for scenario in ["miniberry-30", "miniberry-30-tomato"]:
    _, environments[scenario] = create_wbfe(precompute_environment(scenario))
    environments[scenario].proceed(exp_bench["time-start-environment"])
PROBABILISTIC = ["gp-indicator", "occupancy", "mrf", "epidemic-pf", "cnn"]
fig, axes = plt.subplots(1, 2, figsize=(8, 3.8), sharey=True)
for ax, sampler in zip(axes, ["random-500", "random-waypoint-500"]):
    for run in PROBABILISTIC:
        probabilities, diseased = [], []
        for scenario, environment in environments.items():
            mask = environment.my_tomato_mask
            for repetition in range(exp_bench["repetitions"]):
                with open(pathlib.Path(bench_dir, "observations", f"{scenario}-{sampler}-{repetition}.pickle"), "rb") as f:
                    observations = pickle.load(f)
                estimator = make_estimator(run, geo)
                for observation in observations:
                    estimator.add_observation(observation)
                estimator.proceed(1)
                probabilities.append(estimator.im_tylcv.probability[mask])
                diseased.append(environment.tylcv.value[mask] < DISEASED)
        p, d = np.concatenate(probabilities), np.concatenate(diseased)
        bins = np.minimum((p * 10).astype(int), 9)
        shown = [b for b in range(10) if np.sum(bins == b) >= 20]
        ax.plot([p[bins == b].mean() for b in shown], [d[bins == b].mean() for b in shown], marker="o", label=run)
    ax.plot([0, 1], [0, 1], color="black", linewidth=0.8)
    ax.set_xlabel("predicted probability of disease")
    ax.set_title(SAMPLERS[sampler])
axes[0].set_ylabel("observed frequency")
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.savefig(pathlib.Path(benchmark_dir, "estimators-reliability.pdf"))
